# 01a — Load and clean 3-D hologram stacks

Load one detector acquisition without collapsing its frame axis, subtract an averaged and optionally linearly fitted dark, inspect its intensities, threshold the corrected frames, average them, and save one compact 2-D image using its acquisition ID.

In [59]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt


import numpy as np

import numpy as np


def fast_stack_azimuthal_average(
    images,
    center=None,
    mask=None,
    bin_width=1.0,
):
    """
    Compute azimuthal averages of:

        1. mean(images, axis=0)
        2. std(images, axis=0)

    Parameters
    ----------
    images : ndarray or list of ndarray
        3D array with shape (N, Ny, Nx), or a list of 3D arrays.
        For a list, all arrays are concatenated along axis 0.

    center : tuple (y0, x0), optional
        Center of radial averaging in pixel coordinates.
        Default = geometric center.

    mask : 2D ndarray, optional
        Pixels are included only where mask == 0.

    bin_width : float
        Width of radial bins in pixels.

    Returns
    -------
    r : ndarray
        Radial-bin centers in pixels.

    mean_radial : ndarray
        Azimuthal average of np.mean(images, axis=0).

    std_radial : ndarray
        Azimuthal average of np.std(images, axis=0).

    counts : ndarray
        Number of valid pixels in each radial bin.
    """

    # ---------------------------------------------------------
    # Combine stacks
    # ---------------------------------------------------------

    if isinstance(images, (list, tuple)):
        images = np.concatenate(images, axis=0)

    images = np.asarray(images)

    if images.ndim != 3:
        raise ValueError(
            "images must have shape (N, Ny, Nx), "
            "or be a list of 3D arrays."
        )

    n_images, ny, nx = images.shape

    # ---------------------------------------------------------
    # Mean and standard deviation along stack axis
    # ---------------------------------------------------------

    mean_image = np.nanmean(images, axis=0)
    std_image = np.nanstd(images, axis=0)

    # ---------------------------------------------------------
    # Radial coordinate
    # ---------------------------------------------------------

    if center is None:
        y0 = (ny - 1) / 2
        x0 = (nx - 1) / 2
    else:
        y0, x0 = center

    yy, xx = np.indices((ny, nx))

    radius = np.sqrt(
        (yy - y0)**2
        + (xx - x0)**2
    )

    radial_bin = np.floor(
        radius / bin_width
    ).astype(np.int32)

    # ---------------------------------------------------------
    # Valid pixels
    # ---------------------------------------------------------

    valid = (
        np.isfinite(mean_image)
        & np.isfinite(std_image)
    )

    if mask is not None:

        mask = np.asarray(mask)

        if mask.shape != (ny, nx):
            raise ValueError(
                f"mask shape {mask.shape} does not match "
                f"image shape {(ny, nx)}"
            )

        valid &= (mask == 0)

    bins = radial_bin[valid].ravel()

    mean_values = mean_image[valid].ravel()
    std_values = std_image[valid].ravel()

    nr = bins.max() + 1

    # ---------------------------------------------------------
    # Number of pixels per radial bin
    # ---------------------------------------------------------

    counts = np.bincount(
        bins,
        minlength=nr
    ).astype(float)

    # ---------------------------------------------------------
    # Azimuthal average of mean image
    # ---------------------------------------------------------

    mean_sum = np.bincount(
        bins,
        weights=mean_values,
        minlength=nr
    )

    mean_radial = np.full(nr, np.nan)

    good = counts > 0

    mean_radial[good] = (
        mean_sum[good]
        / counts[good]
    )

    # ---------------------------------------------------------
    # Azimuthal average of standard-deviation image
    # ---------------------------------------------------------

    std_sum = np.bincount(
        bins,
        weights=std_values,
        minlength=nr
    )

    std_radial = np.full(nr, np.nan)

    std_radial[good] = (
        std_sum[good]
        / counts[good]
    )

    # ---------------------------------------------------------
    # Radius = center of each radial bin
    # ---------------------------------------------------------

    r = (
        np.arange(nr) + 0.5
    ) * bin_width

    return r, mean_radial, std_radial, counts

def fit_horizontal_band(
    pattern,
    mask=None,
    nedge=10,
    nstart=0,
    band_center=None,
    band_width=None,
    band_edge=None,
    band_amplitude=None,
    polynomial_order=2,
    plot=True,
):
    """
    Fit a horizontal detector-band artifact using the left/right edges
    of a 2D scattering pattern.

    Model
    -----
    measured profile =
        polynomial background
        + negative smoothed-box artifact

    Only pixels where mask == 0 are used.

    Parameters
    ----------
    pattern : 2D ndarray
        Scattering pattern.

    mask : 2D ndarray or None
        Pixels are used only where mask == 0.
        If None, all finite pixels are used.

    nedge : int
        Number of columns averaged on each side.

    nstart : int
        Number of pixels skipped from the outer detector edges.

    band_center : float, optional
        Initial estimate of band center [pixels].
        Default = image center.

    band_width : float, optional
        Initial estimate of band width [pixels].

    band_edge : float, optional
        Initial estimate of edge smoothing [pixels].

    band_amplitude : float, optional
        Initial estimate of the positive depth of the negative band.

    polynomial_order : int
        Order of polynomial background.
        Default = 2.

    plot : bool
        If True, show diagnostic plots.

    Returns
    -------
    band_2d : ndarray
        Estimated NEGATIVE horizontal-band artifact,
        same shape as pattern.

        Correct with:

            corrected = pattern - band_2d

    polynomial_2d : ndarray
        Fitted polynomial background tiled horizontally
        to match pattern.shape.

    fit_info : dict
        Fitted parameters and intermediate 1D profiles.
    """

    pattern = np.asarray(pattern, dtype=float)

    if pattern.ndim != 2:
        raise ValueError("pattern must be a 2D array.")

    if polynomial_order < 0:
        raise ValueError("polynomial_order must be >= 0.")

    ny, nx = pattern.shape
    y = np.arange(ny, dtype=float)

    # Centered coordinate improves polynomial conditioning
    yc = ny / 2
    yy = y - yc

    # =========================================================
    # Mask
    # =========================================================

    if mask is None:
        invalid = ~np.isfinite(pattern)

    else:
        mask = np.asarray(mask)

        if mask.shape != pattern.shape:
            raise ValueError(
                f"mask shape {mask.shape} does not match "
                f"pattern shape {pattern.shape}"
            )

        invalid = (mask != 0) | ~np.isfinite(pattern)

    # =========================================================
    # Extract left/right edge regions
    # =========================================================

    left_data = pattern[:, nstart:nstart + nedge].copy()
    left_invalid = invalid[:, nstart:nstart + nedge]

    if nstart == 0:
        right_data = pattern[:, -nedge:].copy()
        right_invalid = invalid[:, -nedge:]
    else:
        right_data = pattern[:, -nedge - nstart:-nstart].copy()
        right_invalid = invalid[:, -nedge - nstart:-nstart]

    left_data[left_invalid] = np.nan
    right_data[right_invalid] = np.nan

    left_profile = np.nanmean(left_data, axis=1)
    right_profile = np.nanmean(right_data, axis=1)

    # Allow one side to contribute if the other is masked
    profile = np.nanmean(
        np.stack([left_profile, right_profile]),
        axis=0
    )

    # =========================================================
    # Model components
    # =========================================================

    def smooth_box(y, amplitude, center, width, edge_sigma):

        y1 = center - width / 2
        y2 = center + width / 2

        return amplitude / 2 * (
            erf((y - y1) / (np.sqrt(2) * edge_sigma))
            - erf((y - y2) / (np.sqrt(2) * edge_sigma))
        )

    def polynomial_background(y, *coeffs):

        x = y - yc

        result = np.zeros_like(y, dtype=float)

        for power, coefficient in enumerate(coeffs):
            result += coefficient * x**power

        return result

    def model(y, *params):

        coeffs = params[:polynomial_order + 1]

        band_amp = params[polynomial_order + 1]
        band_center_fit = params[polynomial_order + 2]
        band_width_fit = params[polynomial_order + 3]
        band_edge_fit = params[polynomial_order + 4]

        polynomial = polynomial_background(
            y,
            *coeffs
        )

        band = smooth_box(
            y,
            band_amp,
            band_center_fit,
            band_width_fit,
            band_edge_fit,
        )

        return polynomial - band

    # =========================================================
    # Initial guesses
    # =========================================================

    if band_center is None:
        band_center = ny / 2

    if band_width is None:
        band_width = ny * 0.08

    if band_edge is None:
        band_edge = max(2, band_width / 10)

    valid = np.isfinite(profile)

    if not np.any(valid):
        raise ValueError("No valid unmasked pixels available for fitting.")

    # First estimate polynomial coefficients without worrying
    # too much about the band.
    #
    # np.polyfit returns highest power first, so reverse it
    # to match c0, c1, c2, ...
    poly_initial_high_to_low = np.polyfit(
        yy[valid],
        profile[valid],
        polynomial_order
    )

    poly_initial = poly_initial_high_to_low[::-1]

    # ---------------------------------------------------------
    # Initial band amplitude
    # ---------------------------------------------------------

    if band_amplitude is None:

        distance = np.abs(y - band_center)

        inside = (
            (distance < band_width / 2)
            & valid
        )

        outside = (
            (distance > band_width)
            & (distance < 2 * band_width)
            & valid
        )

        if np.any(inside) and np.any(outside):

            band_amplitude = (
                np.nanmedian(profile[outside])
                - np.nanmedian(profile[inside])
            )

        else:

            band_amplitude = 0.05 * (
                np.nanmax(profile)
                - np.nanmin(profile)
            )

        band_amplitude = max(
            band_amplitude,
            1e-12
        )

    p0 = list(poly_initial) + [
        band_amplitude,
        band_center,
        band_width,
        band_edge,
    ]

    # =========================================================
    # Bounds
    # =========================================================

    lower = (
        [-np.inf] * (polynomial_order + 1)
        + [
            0,      # band amplitude
            0,      # band center
            1,      # band width
            0.2,    # band edge
        ]
    )

    upper = (
        [np.inf] * (polynomial_order + 1)
        + [
            np.inf,
            ny,
            ny,
            ny / 2,
        ]
    )

    # =========================================================
    # Fit
    # =========================================================

    if np.count_nonzero(valid) < len(p0):
        raise ValueError(
            "Not enough unmasked data points to perform the fit."
        )

    popt, pcov = curve_fit(
        model,
        y[valid],
        profile[valid],
        p0=p0,
        bounds=(lower, upper),
        maxfev=50000,
    )

    # =========================================================
    # Extract fitted components
    # =========================================================

    poly_coeffs = popt[:polynomial_order + 1]

    band_index = polynomial_order + 1

    fitted_polynomial = polynomial_background(
        y,
        *poly_coeffs
    )

    fitted_band_positive = smooth_box(
        y,
        *popt[band_index:band_index + 4]
    )

    # Actual detector contribution is negative
    band_1d = -fitted_band_positive

    fitted_total = (
        fitted_polynomial
        + band_1d
    )

    # =========================================================
    # Convert to 2D
    # =========================================================

    band_2d = np.repeat(
        band_1d[:, None],
        nx,
        axis=1
    )

    polynomial_2d = np.repeat(
        fitted_polynomial[:, None],
        nx,
        axis=1
    )

    # =========================================================
    # Diagnostic plots
    # =========================================================

    if plot:

        plt.figure(figsize=(10, 5))

        plt.plot(
            y,
            left_profile,
            alpha=0.4,
            label="Left edge"
        )

        plt.plot(
            y,
            right_profile,
            alpha=0.4,
            label="Right edge"
        )

        plt.plot(
            y[valid],
            profile[valid],
            ".",
            label="Data used for fit"
        )

        plt.plot(
            y,
            fitted_total,
            linewidth=2.5,
            label="Complete fit"
        )

        plt.plot(
            y,
            fitted_polynomial,
            "--",
            linewidth=2,
            label=f"Polynomial background (order {polynomial_order})"
        )

        plt.fill_between(
            y,
            fitted_polynomial,
            fitted_total,
            alpha=0.2,
            label="Band artifact"
        )

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Horizontal-band fit")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Individual components
        # -----------------------------------------------------

        plt.figure(figsize=(10, 4))

        plt.plot(
            y,
            fitted_polynomial,
            label="Polynomial background"
        )

        plt.plot(
            y,
            band_1d,
            label="Band artifact"
        )

        plt.axhline(0, linewidth=1)

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Extracted fit components")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Parameters
        # -----------------------------------------------------

        print("\nFitted polynomial:")
        for i, coeff in enumerate(poly_coeffs):
            print(f"  c{i} = {coeff:.6g}")

        print("\nFitted horizontal band:")
        print(f"  amplitude  = {popt[band_index]:.6g}")
        print(f"  center     = {popt[band_index + 1]:.2f} px")
        print(f"  width      = {popt[band_index + 2]:.2f} px")
        print(f"  edge sigma = {popt[band_index + 3]:.2f} px")

        print(
            f"\nRows used in fit: "
            f"{np.count_nonzero(valid)}/{ny}"
        )

    # =========================================================
    # Output information
    # =========================================================

    polynomial_parameters = {
        f"c{i}": value
        for i, value in enumerate(poly_coeffs)
    }

    fit_info = {
        "polynomial_order": polynomial_order,
        "polynomial_parameters": polynomial_parameters,

        "band_amplitude": popt[band_index],
        "band_center": popt[band_index + 1],
        "band_width": popt[band_index + 2],
        "band_edge": popt[band_index + 3],

        "covariance": pcov,

        "left_profile": left_profile,
        "right_profile": right_profile,
        "profile": profile,
        "valid": valid,

        "polynomial_1d": fitted_polynomial,
        "band_1d": band_1d,
        "fit_profile": fitted_total,
    }

    return band_2d, polynomial_2d, fit_info
    
    

from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())
    
BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)

from data_loading import SextantsNexusLoader, load_processing
print("Base folder:", BASEFOLDER)


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf
import CCI_core as cci


%matplotlib qt

Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW


## Configuration

In [39]:

RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"

IMAGE_ID = 569
DARK_IDS = [574]


## Load the 3-D arrays

In [40]:
loader = SextantsNexusLoader(RAW_FOLDER)

# load_processing returns the simple 2-D average and the full 3-D stack.
blind_average, stack = load_processing(loader, IMAGE_ID)

# The dark is also a 3-D acquisition. Use its simple frame average.
dark_reference, dark_stack = load_processing(loader, DARK_IDS)
dark_reference = np.asarray(dark_reference, dtype=float)
print(f"Loaded ID {IMAGE_ID}: stack {stack.shape}, average {blind_average.shape}")
print("Dark stack:", dark_stack.shape, "dark average:", dark_reference.shape)

Loaded ID 569: stack (101, 2048, 2048), average (2048, 2048)
Dark stack: (11, 2048, 2048) dark average: (2048, 2048)


## Dark subtraction

Fit `frame ≈ scale × dark + offset` on low-intensity pixels from a small selected detector region. The fitted dark background is then subtracted from every pixel of the complete frame.

In [72]:
mask_detector=1.*(plt.imread("processed/mask_pixels/mask_detector.png")[:,:,0]==1)
mask_pixel=1.*(plt.imread("processed/mask_pixels/mask_beamstop_569.png")[:,:,0]==1)
mask_pixel = (wf.center_image(mask_pixel, [1024+1024-997,1024+1024-1040], cci) > 0.5).astype(np.uint8)



plt.close("all")


# Dark-rescaling controls are here because they apply to this operation.
FIT_DARK_LINEAR = True
DARK_FIT_PERCENTILE = 100
DARK_FIT_ROWS = slice(0, 600)
DARK_FIT_COLUMNS = slice(0, 200)
DARK_FIT_STRIDE = 1  # Optional subsampling within the selected region.



def subtract_fitted_dark(frame, dark, rows, columns, percentile=30, stride=1):
    # Estimate scale and offset only from the selected detector region.
    sample_image = frame[rows, columns][::stride, ::stride].ravel().astype(float)
    sample_dark = dark[rows, columns][::stride, ::stride].ravel().astype(float)
    sample_mask_detector=mask_detector[rows, columns][::stride, ::stride].ravel().astype(float)
    valid = np.isfinite(sample_image) & np.isfinite(sample_dark)&(sample_mask_detector==0)
    limit = np.percentile(sample_image[valid], percentile)
    valid &= sample_image <= limit
    if valid.sum() < 2 or np.ptp(sample_dark[valid]) == 0:
        scale, offset = 1.0, 0.0
    else:
        scale, offset = np.polyfit(sample_dark[valid], sample_image[valid], 1)
    corrected = frame - (scale * dark + offset)
    return corrected.astype(np.float32), float(scale), float(offset)

corrected_frames = []
dark_fits = []
for frame in stack:
    if FIT_DARK_LINEAR:
        corrected, scale, offset = subtract_fitted_dark(
            frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
            DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
        )
    else:
        corrected, scale, offset = frame - dark_reference, 1.0, 0.0
    corrected_frames.append(corrected)
    dark_fits.append((scale, offset))
corrected_stack = np.stack(corrected_frames)
dark_fits = np.asarray(dark_fits)
print("Dark scale mean:", dark_fits[:, 0].mean(),
      "offset mean:", dark_fits[:, 1].mean())

Dark scale mean: 0.8689045252658615 offset mean: 23.111283595385398


In [73]:
cimshow(np.clip(np.average(corrected_stack, axis=0)*(1-np.clip(mask_detector+mask_pixel,0,1)), None,None))



interactive(children=(FloatRangeSlider(value=(-5.392873998165131, 7655.461746094254), description='contrast', …

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [74]:
r, Imean, Istd, counts = fast_stack_azimuthal_average(
    corrected_stack,
    center=(1024,1024),
    mask=np.clip(mask_detector+mask_pixel,0,1),
    bin_width=2,
)

In [77]:
fig,(ax,ax2)=plt.subplots(2,1)


ax.plot(r, Imean)

ax.fill_between(
    r,
    Imean - Istd,
    Imean + Istd,
    alpha=0.3
)
ax2.plot(r, Imean/Istd)
ax.set_yscale("log")
ax2.set_yscale("log")
ax.set_xlabel("Radial distance [pixel]")
ax.set_ylabel("Intensity")


Text(0, 0.5, 'Intensity')

In [78]:
plt.close("all")
fig,ax=plt.subplots()
roiii=np.s_[1500:1800,300:800]
_=ax.hist(corrected_stack[0][roiii][mask_detector[roiii]==0].flatten(), range=(0,300), bins=600)

In [79]:
# calculate and subtract the band

cimshow(np.clip(np.average(corrected_stack, axis=0), None, 99.0))

interactive(children=(FloatRangeSlider(value=(-18.6893127784729, 99.0), description='contrast', layout=Layout(…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [80]:
plt.close("all")

band_do=True
nedge=20
nstart=0


mask_detector2=mask_detector.copy()
mask_detector2[-200:,:]=1
mask_detector2[:20,:]=1
if band_do:
    band, gaussian,fit = fit_horizontal_band(
        np.average(corrected_stack, axis=0),
        nedge=nedge,nstart=nstart,
        band_center=1024,
        band_width=80,
        band_edge=11,
        mask=mask_detector2
    )


Fitted polynomial:
  c0 = -0.272469
  c1 = 2.32784e-05
  c2 = -1.36999e-07

Fitted horizontal band:
  amplitude  = 0.767207
  center     = 997.29 px
  width      = 178.69 px
  edge sigma = 7.29 px

Rows used in fit: 1760/2048


/tmp/ipykernel_197939/2980801492.py:314: RuntimeWarning: Mean of empty slice
  left_profile = np.nanmean(left_data, axis=1)
/tmp/ipykernel_197939/2980801492.py:315: RuntimeWarning: Mean of empty slice
  right_profile = np.nanmean(right_data, axis=1)
/tmp/ipykernel_197939/2980801492.py:318: RuntimeWarning: Mean of empty slice
  profile = np.nanmean(


In [82]:
plt.close("all")
if band_do:
    cimshow(np.clip(np.average(corrected_stack, axis=0)-band-gaussian, None, 99.0))

interactive(children=(FloatRangeSlider(value=(-18.279283348306404, 99.0), description='contrast', layout=Layou…

In [83]:
if band_do:
    corrected_stack=corrected_stack-band-gaussian

## Inspect the dark rescaling fit

The scatter plot uses the first frame of the selected image. Grey points are all finite sampled pixels, blue points are the pixels used for the linear fit, and the red line is the fitted dark background.

In [84]:
# Show the same pixels and selection used for the first frame's fit.
image_values = stack[0, DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
dark_values = dark_reference[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
finite = np.isfinite(image_values) & np.isfinite(dark_values)
fit_limit = np.percentile(image_values[finite], DARK_FIT_PERCENTILE)
used = finite & (image_values <= fit_limit)
scale, offset = dark_fits[0]

fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(dark_values[finite], image_values[finite], s=3, alpha=0.08,
             color="0.4", rasterized=True, label="sampled pixels")
axis.scatter(dark_values[used], image_values[used], s=4, alpha=0.25,
             color="tab:blue", rasterized=True, label="pixels used for fit")
x_line = np.linspace(dark_values[used].min(), dark_values[used].max(), 200)
axis.plot(x_line, scale * x_line + offset, color="red", linewidth=2,
          label=f"fit: y = {scale:.4g} x + {offset:.4g}")
axis.set_title(f"ID {IMAGE_ID}: first-frame dark fit")
axis.set_xlabel("Dark-reference intensity")
axis.set_ylabel("Raw-frame intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Inspect corrected intensities

Choose the thresholds in the next cell, then rerun that cell and the cleaning cell below.

In [85]:
plt.close("all")
fig,ax=plt.subplots()
roiii=np.s_[:,:]
_=ax.hist(corrected_stack[0][roiii][mask_detector[roiii]==0].flatten(), range=(0,200), bins=600)

In [86]:
plt.close("all")


# Edit these values while inspecting the histograms below.
INTENSITY_THRESHOLD = 50.0
HISTOGRAM_RANGE = (-20, 80)
HISTOGRAM_BINS = 400
PHOTON_VIEW_ROWS = slice(402, 900)
PHOTON_VIEW_COLUMNS = slice(420, 900)
gauss_do=False


representative_image = corrected_stack[0]
fig, axis = plt.subplots(figsize=(6, 4))
axis.hist(representative_image.ravel(), bins=HISTOGRAM_BINS, range=HISTOGRAM_RANGE, histtype="step")
axis.axvline(INTENSITY_THRESHOLD, color="red", linestyle="--", label="threshold")
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Intensity")
axis.set_ylabel("Pixel count")
axis.set_xlim(*HISTOGRAM_RANGE)
axis.legend()
axis.set_yscale("log")
plt.tight_layout()
plt.show()

import scipy
def gauss(image, sigma=3):
    return scipy.ndimage.gaussian_filter(image, sigma=sigma)
    
# Inspect the same first frames on the scale of individual photon events.
fig, axis = plt.subplots(figsize=(6, 5))
photon_view = corrected_stack[0, PHOTON_VIEW_ROWS, PHOTON_VIEW_COLUMNS]
if gauss_do:
    image = axis.imshow(
    photon_view*(gauss(photon_view, 2)>(INTENSITY_THRESHOLD/1.5)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
else:
    image = axis.imshow(
    photon_view*((photon_view)>(INTENSITY_THRESHOLD)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
    
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Column in selected region")
axis.set_ylabel("Row in selected region")
fig.colorbar(image, ax=axis, label="Corrected intensity")
plt.tight_layout()
plt.show()

## Threshold, clip, and average

In [15]:
plt.close("all")


In [87]:
# Subtract the chosen threshold from every frame, clip negatives, then average.
if gauss:
    blurred_stack=corrected_stack.copy()
    for i in range(blurred_stack.shape[0]):
        blurred_stack[i]=corrected_stack[i]*(gauss(corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
    cleaned_stack = np.clip(blurred_stack - 0*float(INTENSITY_THRESHOLD), 0, None)
else:
    blurred_stack=corrected_stack.copy()
    cleaned_stack = np.clip(blurred_stack - float(INTENSITY_THRESHOLD), 0, None)


cleaned_average = np.mean(cleaned_stack, axis=0, dtype=np.float64)
print("Threshold:", INTENSITY_THRESHOLD, "average shape:", cleaned_average.shape)

# Compare the blind average returned by load_processing with the cleaned average.
# Each column uses one shared linear color scale so the change is directly visible.
AVERAGE_DISPLAY_PERCENTILES = (1, 10.9)

comparison_values = np.concatenate((blind_average.ravel(), cleaned_average.ravel()))
comparison_values = comparison_values[np.isfinite(comparison_values)]
vmin, vmax = np.percentile(comparison_values, AVERAGE_DISPLAY_PERCENTILES)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))


for axis, image, description in zip(
    axes,
    (blind_average, cleaned_average),
    ("blind load_processing average", "cleaned average"),
):
    shown = axis.imshow(image, vmin=vmin, vmax=vmax, cmap="viridis")
    axis.set_title(f"ID {IMAGE_ID}: {description} (linear scale)")
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, label="Average intensity")
plt.tight_layout()
plt.show()

Threshold: 50.0 average shape: (2048, 2048)


In [88]:
temp=np.clip(cleaned_average, None, 1000)
cimshow(temp)

interactive(children=(FloatRangeSlider(value=(0.0, 1000.0), description='contrast', layout=Layout(width='500px…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [89]:
cimshow(np.abs(fth.reconstruct(cleaned_average)))

interactive(children=(FloatRangeSlider(value=(0.00033524486243357204, 4.274408508473123), description='contras…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [94]:
temp=np.clip(blind_average, None, 10000)
cimshow(temp)

interactive(children=(FloatRangeSlider(value=(165.16831683168317, 9701.878811881683), description='contrast', …

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [91]:
BASEFOLDER="/home/experiences/sextants/com-sextants/SEXT_NEW"

## Save one cleaned average per acquisition

In [92]:
plt.close("all")

OUTPUT_FOLDER = BASEFOLDER + "/processed/" + "cleaned_acquisitions"
USER = "rb"

setup_metadata = loader.load(IMAGE_ID).metadata
output_file = OUTPUT_FOLDER + f"cleaned_ImId_{IMAGE_ID:04d}_{USER}.npz"
np.savez_compressed(
    output_file,
    image=cleaned_average,
    blind_average=blind_average,
    image_id=np.asarray(IMAGE_ID, dtype=int),
    dark_ids=np.asarray(DARK_IDS, dtype=int),
    threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
    dark_fits=dark_fits,
    energy_eV=float(setup_metadata["energy_eV"]),
    ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
    px_size_m=11.0e-6,
)
print(f"Saved ID {IMAGE_ID}: {output_file}")

Saved ID 569: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitionscleaned_ImId_0569_rb.npz


In [95]:
plt.close("all")

# After tuning the parameters above, put the remaining image IDs here.
# This repeats loading, dark subtraction, thresholding, averaging, and saving
# without producing diagnostic plots. An empty list does nothing.
BATCH_IMAGE_IDS = list(np.arange(609,613) )+list(np.arange(589,602) )+list(np.arange(575,589) )

for batch_image_id in BATCH_IMAGE_IDS:
    plt.close("all")
    
    batch_blind_average, batch_stack = load_processing(loader, batch_image_id)

    batch_corrected_frames = []
    batch_dark_fits = []
    for frame in batch_stack:
        if FIT_DARK_LINEAR:
            corrected, scale, offset = subtract_fitted_dark(
                frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
                DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
            )
        else:
            corrected, scale, offset = frame - dark_reference, 1.0, 0.0
        batch_corrected_frames.append(corrected)
        batch_dark_fits.append((scale, offset))

    batch_corrected_stack = np.stack(batch_corrected_frames)


    if band_do:
        band, gaussian,fit = fit_horizontal_band(
        np.average(batch_corrected_stack, axis=0),mask=mask_detector2,
        nedge=nedge,nstart=nstart,
        band_center=1024,
        band_width=80,
        band_edge=11,plot=False
        )
        batch_corrected_stack=batch_corrected_stack-band-gaussian

    if gauss_do:
        blurred_stack=batch_corrected_stack.copy()
        for i in range(blurred_stack.shape[0]):
            blurred_stack[i]=batch_corrected_stack[i]*(gauss(batch_corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
        batch_cleaned_stack = np.clip(blurred_stack- 0*float(INTENSITY_THRESHOLD), 0, None)
    else:
        blurred_stack=batch_corrected_stack.copy()
        batch_cleaned_stack = np.clip(blurred_stack- float(INTENSITY_THRESHOLD), 0, None)

    batch_cleaned_average = np.mean(
        batch_cleaned_stack, axis=0, dtype=np.float64
    )
    batch_output_file = (
        OUTPUT_FOLDER + f"/cleaned_ImId_{batch_image_id:04d}_{USER}.npz"
    )
    np.savez_compressed(
        batch_output_file,
        image=batch_cleaned_average,
        blind_average=batch_blind_average,
        image_id=np.asarray(batch_image_id, dtype=int),
        dark_ids=np.asarray(DARK_IDS, dtype=int),
        threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
        dark_fits=np.asarray(batch_dark_fits),
        energy_eV=float(setup_metadata["energy_eV"]),
        ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
        px_size_m=11.0e-6,
    )
    print(f"Saved ID {batch_image_id}: {batch_output_file}")

print("fine-tuned im_id:", IMAGE_ID)
print("batch im_ids:", BATCH_IMAGE_IDS)
print("dark_ids:", DARK_IDS)

/tmp/ipykernel_197939/2980801492.py:314: RuntimeWarning: Mean of empty slice
  left_profile = np.nanmean(left_data, axis=1)
/tmp/ipykernel_197939/2980801492.py:315: RuntimeWarning: Mean of empty slice
  right_profile = np.nanmean(right_data, axis=1)
/tmp/ipykernel_197939/2980801492.py:318: RuntimeWarning: Mean of empty slice
  profile = np.nanmean(


Saved ID 609: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0609_rb.npz
Saved ID 610: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0610_rb.npz
Saved ID 611: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0611_rb.npz


ValueError: CCD distance dataset /scan_0612/scan_data/data_03 is empty or invalid